In [1]:
import datetime as dt
import json
import os
import textwrap
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Setup file paths
logs_dir = "../dataset/"
json_file_path = os.path.join(logs_dir, "xgb_permutation_results.json")

now = dt.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
pdf_output_path = os.path.join(
    logs_dir, f"xgb_permutations_report_{now}.pdf"
)

# Load JSON results
if not os.path.exists(json_file_path):
    print(f"Error: Could not find '{json_file_path}'")
    exit()

with open(json_file_path, "r", encoding="utf-8") as f:
    payload = json.load(f)

instances = payload.get("instances", [])
dataset_keys = [
    ("raw_data_results", "Raw Data"),
    ("pruned_data_results", "Pruned Data"),
    ("pca_data_results", "PCA Data"),
]


def render_confusion_matrix(ax, cm_dict, title):
    """Helper to reconstruct and plot a normalized confusion matrix heatmap."""
    labels = list(cm_dict.keys())
    matrix_size = len(labels)
    cm_normalized = np.zeros((matrix_size, matrix_size))

    for i, actual in enumerate(labels):
        for j, pred in enumerate(labels):
            cm_normalized[i, j] = cm_dict[actual][pred]

    annot_labels = np.array(
        [
            [f"{val:.1f}%" if val > 0 else "0" for val in row]
            for row in cm_normalized
        ]
    )

    sns.heatmap(
        cm_normalized,
        annot=annot_labels,
        fmt="",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        cbar=False,
        vmin=0,
        vmax=100,
        ax=ax,
        annot_kws={"size": 6},
    )

    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    ax.set_xlabel("Predicted Activity", fontsize=8, fontweight="bold")
    ax.set_ylabel("Actual Activity", fontsize=8, fontweight="bold")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=7)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)


# Open multi-page PDF document context
with PdfPages(pdf_output_path) as pdf:
    for instance in instances:
        instance_id = instance.get("instance_id")
        timestamp = instance.get("timestamp", "N/A")
        hyperparams = instance.get("parameters", {})

        # Figure set to landscape (12 x 8.5 inches) for 3 side-by-side heatmaps
        fig = plt.figure(figsize=(12, 8.5))
        gs = fig.add_gridspec(
            2, 3, height_ratios=[1.3, 1.0], hspace=0.38, wspace=0.28
        )

        # Title Block
        plt.suptitle(
            f"Hyperparameter Set #{instance_id} Evaluation Report | Executed: {timestamp}",
            fontsize=13,
            fontweight="bold",
            y=0.97,
        )

        # -------------------------------------------------------------
        # Row 1: Confusion Matrices for Raw, Pruned, and PCA Datasets
        # -------------------------------------------------------------
        metrics_summary = []

        for col_idx, (ds_key, ds_name) in enumerate(dataset_keys):
            ax = fig.add_subplot(gs[0, col_idx])
            ds_data = instance.get(ds_key, {})
            final_res = ds_data.get("final_results", {})

            cm_dict = final_res.get("confusion_matrix_pct", {})
            acc = final_res.get("accuracy", 0.0)
            f1 = final_res.get("f1_macro", 0.0)
            mean_acc = final_res.get("mean_fold_accuracy", 0.0)
            std_acc = final_res.get("std_fold_accuracy", 0.0)

            # Accumulate metrics for the summary table
            metrics_summary.append(
                [
                    ds_name,
                    f"{acc:.4f} ({acc * 100:.2f}%)",
                    f"{mean_acc:.4f} ± {std_acc:.4f}",
                    f"{f1:.4f}",
                    f"{final_res.get('precision_macro', 0.0):.4f}",
                    f"{final_res.get('recall_macro', 0.0):.4f}",
                ]
            )

            plot_title = (
                f"{ds_name}\nAcc: {acc * 100:.2f}% | Macro F1: {f1:.4f}"
            )
            render_confusion_matrix(ax, cm_dict, plot_title)

        # -------------------------------------------------------------
        # Row 2, Col 0: Hyperparameters Table
        # -------------------------------------------------------------
        ax_param = fig.add_subplot(gs[1, 0])
        ax_param.axis("off")

        param_table_data = [
            [str(k), str(v)] for k, v in hyperparams.items()
        ]

        param_table = ax_param.table(
            cellText=param_table_data,
            colLabels=["Hyperparameter", "Value"],
            loc="center",
            cellLoc="left",
        )
        param_table.auto_set_font_size(False)
        param_table.set_fontsize(8)
        param_table.scale(1.0, 1.2)

        # Format header
        for col_idx in range(2):
            cell = param_table[(0, col_idx)]
            cell.set_facecolor("#40466e")
            cell.get_text().set_color("white")
            cell.get_text().set_weight("bold")

        ax_param.set_title(
            "Hyperparameter Configuration",
            fontsize=10,
            fontweight="bold",
            pad=6,
        )

        # -------------------------------------------------------------
        # Row 2, Col 1-2: Multi-Dataset Performance Summary Table
        # -------------------------------------------------------------
        ax_summary = fig.add_subplot(gs[1, 1:])
        ax_summary.axis("off")

        summary_headers = [
            "Dataset Variant",
            "OOF Accuracy",
            "Mean Fold Acc",
            "F1 (Macro)",
            "Precision",
            "Recall",
        ]

        summary_table = ax_summary.table(
            cellText=metrics_summary,
            colLabels=summary_headers,
            loc="center",
            cellLoc="center",
        )
        summary_table.auto_set_font_size(False)
        summary_table.set_fontsize(8.5)
        summary_table.scale(1.0, 1.5)

        # Format header and alternate row shading
        for col_idx in range(len(summary_headers)):
            cell = summary_table[(0, col_idx)]
            cell.set_facecolor("#40466e")
            cell.get_text().set_color("white")
            cell.get_text().set_weight("bold")

        for row_idx in range(1, len(metrics_summary) + 1):
            bg_color = "#f2f2f2" if row_idx % 2 == 0 else "#ffffff"
            for col_idx in range(len(summary_headers)):
                summary_table[(row_idx, col_idx)].set_facecolor(bg_color)

        ax_summary.set_title(
            "Performance Comparison Across Dataset Permutations",
            fontsize=10,
            fontweight="bold",
            pad=6,
        )

        # Save current page to PDF
        pdf.savefig(fig)
        plt.close(fig)

print(
    f"Successfully generated multi-page evaluation PDF at: {pdf_output_path}"
)

Successfully generated multi-page evaluation PDF at: ../dataset/xgb_permutations_report_2026-10-07_20-44-09.pdf
